# Bronze → Silver: dim_sap_customers

**Propósito:** Leer `dbo.dim_sap_customers` del lakehouse Bronze, aplicar transformaciones de calidad y escribir en el lakehouse Silver de forma idempotente.

**Transformaciones aplicadas:**
- `postal_code`: cast a STRING con zero-padding a 5 dígitos (los CP españoles que empiezan por 0 lo necesitan)
- `is_active`: columna derivada (`customer_blocked == 0` → activo)
- Drop de `customer_blocked2` (siempre nula en los datos)
- Trim de columnas string (`name`, `city`, `street`, `name_groups`)
- `customer_blocked` cast a BooleanType para legibilidad
- Columna de auditoría `_silver_load_ts`

**Idempotencia:** `overwrite` + `overwriteSchema=true`. Seguro de re-ejecutar.

In [ ]:
%run ./config

StatementMeta(, 245daefa-e0f3-400a-a5fb-724ef0f77c2d, 3, Finished, Available, Finished, True)

Config cargada → Bronze: lakehouse_poc | Silver: silver_lakehouse_poc | Gold: Gold


In [ ]:


BRONZE_TABLE = f"{DEFAULT_SCHEMA}.dim_sap_customers"
SILVER_TABLE = f"{DEFAULT_SCHEMA}.dim_sap_customers"

StatementMeta(, 245daefa-e0f3-400a-a5fb-724ef0f77c2d, 4, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType
from datetime import datetime

df_bronze = spark.read.table(f"`{BRONZE_LAKEHOUSE}`.{BRONZE_TABLE}")

print(f"Filas leídas desde Bronze: {df_bronze.count()}")
df_bronze.printSchema()

StatementMeta(, 245daefa-e0f3-400a-a5fb-724ef0f77c2d, 5, Finished, Available, Finished, False)

Filas leídas desde Bronze: 104430
root
 |-- customer_id: string (nullable = true)
 |-- country_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- street: string (nullable = true)
 |-- customer_blocked: string (nullable = true)
 |-- customer_blocked2: string (nullable = true)
 |-- tax_number: string (nullable = true)
 |-- category: string (nullable = true)
 |-- center_type: string (nullable = true)
 |-- customer_sales_track: string (nullable = true)
 |-- exclude_transfer_sales_track: string (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- name_groups: string (nullable = true)



In [ ]:
# ─── TRANSFORMACIONES ─────────────────────────────────────────────────────────

df_silver = (
    df_bronze

    # 1. postal_code → STRING con zero-padding a 5 dígitos
    #    Ejemplo: 8908 → '08908' (CP catalanes y otros que empiezan por 0)
    .withColumn(
        "postal_code",
        F.lpad(F.col("postal_code").cast("string"), 5, "0")
    )

    # 2. customer_blocked: int 0/1 → boolean
    .withColumn("customer_blocked", F.col("customer_blocked").cast(BooleanType()))

    # 3. Columna derivada: is_active (inverso de customer_blocked)
    .withColumn("is_active", ~F.col("customer_blocked"))

    # 4. Drop de columna siempre nula
    .drop("customer_blocked2")

    # 5. Trim de strings
    .withColumn("name",        F.trim(F.col("name")))
    .withColumn("city",        F.trim(F.col("city")))
    .withColumn("street",      F.trim(F.col("street")))
    .withColumn("name_groups", F.trim(F.col("name_groups")))
    .withColumn("region",      F.trim(F.col("region")))
    .withColumn("country",     F.trim(F.col("country")))

    # 6. Auditoría
    .withColumn("_silver_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
)

StatementMeta(, 245daefa-e0f3-400a-a5fb-724ef0f77c2d, 6, Finished, Available, Finished, False)

In [ ]:
# ─── VALIDACIÓN PREVIA A ESCRITURA ────────────────────────────────────────────
row_count     = df_silver.count()
null_ids      = df_silver.filter(F.col("customer_id").isNull()).count()
dup_ids       = df_silver.groupBy("customer_id").count().filter(F.col("count") > 1).count()
blocked_count = df_silver.filter(F.col("customer_blocked") == True).count()
active_count  = df_silver.filter(F.col("is_active") == True).count()

print(f"Filas a escribir en Silver : {row_count}")
print(f"customer_id nulos          : {null_ids}")
print(f"customer_id duplicados     : {dup_ids}")
print(f"Clientes bloqueados        : {blocked_count}")
print(f"Clientes activos           : {active_count}")

assert null_ids == 0,  "ERROR: hay customer_id nulos"
assert dup_ids  == 0,  "ERROR: hay customer_id duplicados en la dimensión"

df_silver.show(3)

StatementMeta(, 245daefa-e0f3-400a-a5fb-724ef0f77c2d, 7, Finished, Available, Finished, False)

Filas a escribir en Silver : 104430
customer_id nulos          : 0
customer_id duplicados     : 0
Clientes bloqueados        : 0
Clientes activos           : 0
+-----------+----------+--------------------+--------------------+-----------+---------+--------------------+----------------+----------+--------+-----------+--------------------+----------------------------+---------+-------+--------------------+---------+--------------------+
|customer_id|country_id|                name|                city|postal_code|region_id|              street|customer_blocked|tax_number|category|center_type|customer_sales_track|exclude_transfer_sales_track|   region|country|         name_groups|is_active|     _silver_load_ts|
+-----------+----------+--------------------+--------------------+-----------+---------+--------------------+----------------+----------+--------+-----------+--------------------+----------------------------+---------+-------+--------------------+---------+--------------------+
|  

In [ ]:
# ─── ESCRITURA IDEMPOTENTE EN SILVER ──────────────────────────────────────────
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")
)

print(f"✓ Tabla {SILVER_LAKEHOUSE}.{SILVER_TABLE} escrita correctamente con {row_count} filas.")

StatementMeta(, 245daefa-e0f3-400a-a5fb-724ef0f77c2d, 8, Finished, Available, Finished, False)

✓ Tabla silver_lakehouse_poc.dbo.dim_sap_customers escrita correctamente con 104430 filas.
